# Telco Customer Churn Prediction
This notebook performs exploratory data analysis, preprocessing, and classification modeling using the synthetic Telco churn dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style('whitegrid')
%matplotlib inline

df = pd.read_csv('telco_customer_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_total = df['TotalCharges'].isna().sum()
if missing_total > 0:
    df.loc[df['TotalCharges'].isna(), 'TotalCharges'] = df.loc[df['TotalCharges'].isna(), 'MonthlyCharges']

print('Dataset shape:', df.shape)
print('Missing values by column:')
print(df.isna().sum())


## Dataset Overview
Inspect the first rows and class balance for churn.

In [ ]:
display(df.head())
display(df.describe(include='all').T)

plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Churn', palette='Set2')
plt.title('Churn Distribution')
plt.show()


## Feature Distributions and Relationships
Analyze numeric trends and churn behavior across key customer segments.

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(df['MonthlyCharges'], kde=True, bins=30, color='steelblue')
plt.title('Monthly Charges Distribution')

plt.subplot(1,2,2)
sns.histplot(df['TotalCharges'], kde=True, bins=30, color='indianred')
plt.title('Total Charges Distribution')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
sns.boxplot(data=df, x='Churn', y='tenure', palette='Set2')
plt.title('Tenure by Churn')
plt.show()


In [ ]:
categorical_features = ['InternetService', 'Contract', 'PaymentMethod']
plt.figure(figsize=(16,10))
for i, col in enumerate(categorical_features, 1):
    plt.subplot(3,1,i)
    sns.countplot(data=df, x=col, hue='Churn', palette='Set2')
    plt.title(f'{col} vs Churn')
    plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Preprocessing
Encode categorical columns and prepare the model dataset.

In [ ]:
df_model = df.copy()
df_model.drop(columns=['customerID'], inplace=True)
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})
df_model['gender'] = df_model['gender'].map({'Male': 0, 'Female': 1})
df_model['Partner'] = df_model['Partner'].map({'No': 0, 'Yes': 1})
df_model['Dependents'] = df_model['Dependents'].map({'No': 0, 'Yes': 1})
df_model['PhoneService'] = df_model['PhoneService'].map({'No': 0, 'Yes': 1})
df_model['PaperlessBilling'] = df_model['PaperlessBilling'].map({'No': 0, 'Yes': 1})

binary_columns = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines']
for col in binary_columns:
    df_model[col] = df_model[col].replace({'No phone service': 'No', 'No internet service': 'No'})
    df_model[col] = df_model[col].map({'No': 0, 'Yes': 1})

df_model = pd.get_dummies(df_model, columns=['InternetService', 'Contract', 'PaymentMethod'], drop_first=True)

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)


## Model Training and Evaluation
Train Logistic Regression and Random Forest classifiers, then compare performance.

In [ ]:
log_clf = LogisticRegression(max_iter=200, random_state=42)
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)

log_clf.fit(X_train, y_train)
rf_clf.fit(X_train, y_train)

for name, model in [('Logistic Regression', log_clf), ('Random Forest', rf_clf)]:
    preds = model.predict(X_test)
    print(f'--- {name} ---')
    print('Accuracy:', accuracy_score(y_test, preds))
    print(classification_report(y_test, preds, digits=4))
    print('Confusion matrix:
', confusion_matrix(y_test, preds))
    print()


## Feature Importance
Inspect which features had the most influence in the Random Forest model.

In [ ]:
feature_importances = pd.Series(rf_clf.feature_importances_, index=X.columns)
feature_importances = feature_importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10,6))
sns.barplot(x=feature_importances.values, y=feature_importances.index, palette='viridis')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()
